# Bank Marketing Subscription Prediction: End-to-End ML & ANN Benchmark
**Author:** Senior Machine Learning & Deep Learning Engineer  
**Dataset:** UCI Bank Marketing Benchmark (`bank-full.csv`, 45,211 rows, 16 features + target `y`)  
**Objective:** Build, evaluate, explain, and compare Traditional ML models vs. Multi-Architecture Artificial Neural Networks (ANNs).

In [ ]:
import os
import sys
# Ensure Keras uses torch backend
os.environ['KERAS_BACKEND'] = 'torch'

# Add root workspace to sys.path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

from src.data_loader import load_raw_data, validate_data_schema, split_dataset
from src.utils import load_config

cfg = load_config('../config/config.yaml')
df = load_raw_data('../data/bank-full.csv', sep=';')
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")
df.head()

## 1. Data Schema & Class Imbalance Analysis

In [ ]:
val_counts = df['y'].value_counts()
sub_rate = (val_counts['yes'] / len(df)) * 100
imbalance_ratio = val_counts['no'] / val_counts['yes']

print(f"Target Counts:\n{val_counts}\n")
print(f"Subscription Rate: {sub_rate:.2f}%")
print(f"Class Imbalance Ratio (No:Yes): {imbalance_ratio:.2f}:1")

pdays_neg1 = (df['pdays'] == -1).sum()
print(f"Clients not previously contacted (pdays = -1): {pdays_neg1:,} ({pdays_neg1/len(df)*100:.2f}%)")

## 2. Automated Exploratory Data Analysis Figures
Visualizing target distributions, numeric features, categorical subscription rates, and correlation matrix.

In [ ]:
fig_dir = '../reports/figures'
for fname in ['target_distribution.png', 'numerical_distributions.png', 'subscription_by_category.png', 'correlation_matrix.png']:
    p = os.path.join(fig_dir, fname)
    if os.path.exists(p):
        print(f"=== Figure: {fname} ===")
        display(Image(filename=p))

## 3. Investigating Target Leakage: The Duration Phenomenon
`duration` has a strong correlation with subscription ($r \approx 0.39$). However, call duration is only known after a call takes place!  
Prior to dialing a lead, duration is unknown. Training a model with duration creates severe target leakage if used for pre-call prioritization.

In [ ]:
dur_p = os.path.join(fig_dir, 'duration_leakage_analysis.png')
if os.path.exists(dur_p):
    display(Image(filename=dur_p))

## 4. Benchmark Model Evaluation & Test-Set Comparison Table
Loading the real measured test evaluation metrics from `reports/model_comparison.csv`.

In [ ]:
comparison_df = pd.read_csv('../reports/model_comparison.csv')
display(comparison_df[['Model', 'Type', 'Features', 'PR-AUC', 'ROC-AUC', 'Accuracy (0.5)', 'F1-Score (0.5)', 'Val-Optimized Thresh', 'F1-Score (Opt Thresh)', 'Recall (Opt Thresh)', 'Brier Score', 'Training Time (s)']])

## 5. ROC Curves, Precision-Recall Curves & Calibration
Comparing the discrimination ability of all 9 trained configurations on the untouched holdout test partition.

In [ ]:
for fname in ['roc_curves.png', 'pr_curves.png', 'metrics_comparison_bar.png', 'calibration_curves.png', 'ann_learning_curves.png']:
    p = os.path.join(fig_dir, fname)
    if os.path.exists(p):
        print(f"=== Evaluation Chart: {fname} ===")
        display(Image(filename=p))

## 6. Model Explainability: SHAP & Permutation Importance
Aggregated feature importance across the 16 original business attributes.

In [ ]:
for fname in ['shap_summary_plot.png', 'aggregated_feature_comparison.png']:
    p = os.path.join(fig_dir, fname)
    if os.path.exists(p):
        display(Image(filename=p))

agg_imp_p = '../reports/aggregated_feature_importance.csv'
if os.path.exists(agg_imp_p):
    agg_df = pd.read_csv(agg_imp_p)
    print("\nAggregated Feature Importance:")
    display(agg_df.head(10))

## 7. Interactive Inference Demo using `BankPredictor`

In [ ]:
from src.predict import BankPredictor

predictor = BankPredictor(config_path='../config/config.yaml')

sample_customer = {
    'age': 42,
    'job': 'management',
    'marital': 'single',
    'education': 'tertiary',
    'default': 'no',
    'balance': 3500,
    'housing': 'no',
    'loan': 'no',
    'contact': 'cellular',
    'day': 18,
    'month': 'sep',
    'duration': 420,
    'campaign': 1,
    'pdays': 90,
    'previous': 2,
    'poutcome': 'success'
}

res_post = predictor.predict(sample_customer, model_name='Advanced ANN', mode='post_call')
print("Post-Call Prediction:", res_post)

sample_pre = dict(sample_customer)
del sample_pre['duration']
res_pre = predictor.predict(sample_pre, model_name='Pre-call XGBoost (No Duration)', mode='pre_call')
print("\nPre-Call Prediction:", res_pre)